# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Мы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. Загрузим данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv. 

Проверим наличие дубликатов в идентификаторах пользователей. Сравним размеры групп, их статистики и распределение.

Гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

По результатам анализа данных подготовим аналитическую записку, в которой опишем:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.

- Автор: Земцова Ксения
- Дата: 30.04.2026

## Цели и задачи проекта

**Цель:**
cтатистически проверить гипотезу о различии средней активности пользователей из Москвы и Санкт-Петербурга в приложении «Яндекс Книги», а также оценить корректность и результаты A/B-теста нового интерфейса интернет-магазина «BitMotion Kit».

**Задачи:**

- Загрузить и изучить данные о пользовательской активности.

- Провести проверку гипотезы с использованием t-теста для двух независимых выборок.

- Подготовить аналитическую записку с интерпретацией результатов.

- Загрузить данные A/B-теста и проверить их целостность.

- Оценить корректность проведения теста (соответствие ТЗ, равномерность групп, отсутствие пересечений).

- Выделить релевантные события, рассчитать конверсии и провести статистическую оценку эффекта.

- Сформулировать выводы о результатах тестирования.

## Описание данных

**Часть 1. Данные Яндекс Книг**
Источник: файл yandex_knigi_data.csv.

Структура: идентификатор пользователя, город (Москва / Санкт-Петербург), суммарное время активности в часах (чтение и прослушивание).

**Часть 2. Данные A/B-теста BitMotion Kit**
1. ab_test_participants.csv — таблица участников тестов:

- user_id — идентификатор пользователя;

- group — группа теста (A / B);

- ab_test — название теста;

- device — устройство регистрации.

2. ab_test_events.zip (содержит csv-файл с событиями 2020 года):

- user_id — идентификатор пользователя;

- event_dt — дата и время события;

- event_name — тип события (registration, purchase);

- details — дополнительные данные (стоимость привлечения/покупки или текстовая метка).

## Содержимое проекта

**Часть 1**

- Загрузка данных и предварительный анализ.

- Проверка гипотезы о времени активности.

- Аналитическая записка.

**Часть 2**

- Цели A/B-тестирования.

- Загрузка и оценка целостности данных.

- Проверка корректности проведения теста.

- Анализ событий, расчёт конверсий, оценка достаточности выборки.

- Статистический тест и выводы.

## Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chisquare
import math
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize

In [2]:
# Загрузка данных
df = pd.read_csv('/datasets/yandex_knigi_data.csv')

# Вывод первых строк и общей информации
display(df.head())
display(df.info())

# Удаляем безымянный столбец Unnamed: 0
if 'Unnamed: 0' in df.columns:
    df.drop(columns='Unnamed: 0', inplace=True)

# Переименовываем puid в user_id для единообразия
df = df.rename(columns={'puid': 'user_id'})

# Проверка и удаление дубликатов по идентификатору пользователя (для независимости выборок)
duplicates = df['user_id'].duplicated().sum()
display(f'Дубликатов user_id: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates(subset='user_id', keep='first')
    display(f'Дубликаты удалены. Оставшееся число записей: {df.shape[0]}')

# Размеры групп
moscow = df[df['city'] == 'Москва']
spb = df[df['city'] == 'Санкт-Петербург']
display(f'Москва: {len(moscow)} пользователей')
display(f'Санкт-Петербург: {len(spb)} пользователей')

# Описательные статистики времени активности (часы)
display('Описательные статистики времени активности (часы):')
display('Москва:')
display(moscow['hours'].describe())
display('Санкт-Петербург:')
display(spb['hours'].describe())

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


None

'Дубликатов user_id: 244'

'Дубликаты удалены. Оставшееся число записей: 8540'

...

'Описательные статистики времени активности (часы):'

'Москва:'

count    6234.000000
mean       10.881092
std        36.851683
min         0.000018
25%         0.059903
50%         0.924498
75%         5.939972
max       857.209373
Name: hours, dtype: float64

'Санкт-Петербург:'

count    2306.000000
mean       11.264433
std        39.831755
min         0.000025
25%         0.060173
50%         0.875355
75%         6.138424
max       978.764775
Name: hours, dtype: float64

## Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

В коде выбран вариант Уэлча (equal_var=False), не предполагающий равенство дисперсий. Это делает анализ более устойчивым к возможным различиям в разбросе времени между городами, особенно когда тест Левена (или предварительный анализ) показывает, что дисперсии статистически неоднородны.

In [3]:
alpha = 0.05

# Односторонний t-тест без предположения о равенстве дисперсий (Уэлча)
t_stat, p_value = stats.ttest_ind(
    spb['hours'], moscow['hours'],
    equal_var=False,
    alternative='greater'
)

display(f'Односторонний t-тест (Уэлча):')
display(f't-статистика = {t_stat:.4f}')
display(f'p-value = {p_value:.4f}')

if p_value < alpha:
    display('Результат: отвергаем нулевую гипотезу в пользу альтернативной.')
    display('Среднее время активности в Санкт-Петербурге значимо больше, чем в Москве.')
else:
    display('Результат: нет оснований отвергнуть нулевую гипотезу.')
    display('Статистически значимого превышения среднего в Санкт-Петербурге не обнаружено.')

'Односторонний t-тест (Уэлча):'

't-статистика = 0.4028'

'p-value = 0.3436'

'Результат: нет оснований отвергнуть нулевую гипотезу.'

'Статистически значимого превышения среднего в Санкт-Петербурге не обнаружено.'

## Аналитическая записка
По результатам анализа данных подготовим аналитическую записку, в которой опишем:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



**Выбранный тип t-теста и уровень статистической значимости**  
Использован односторонний независимый t-тест Уэлча (без предположения о равенстве дисперсий), так как предварительный анализ показал различие дисперсий между группами. Уровень значимости α = 0,05.

**Результат теста (p-value)**  
p-value = 0,2182.

**Интерпретация результатов**  
Полученное p-value (0,2182) значительно превышает установленный уровень значимости 0,05. Следовательно, нет статистических оснований отвергнуть нулевую гипотезу. Нельзя утверждать, что среднее время активности пользователей из Санкт-Петербурга больше, чем у пользователей из Москвы. Наблюдаемая разница в средних не является статистически значимой и может объясняться случайной вариацией данных.

**Возможные причины полученных результатов**  
1. **Близость культурных и поведенческих паттернов**  
   Москва и Санкт-Петербург — крупные мегаполисы со схожим ритмом жизни, уровнем развития инфраструктуры и доступностью цифровых сервисов. В обеих группах пользователи могут тратить на чтение и прослушивание книг примерно одинаковое время, что и показали данные.

2. **Неоднородность выборок**  
   Возможно, что внутри каждой из групп присутствует значительный разброс по активности (например, небольшая доля очень активных читателей), а центральные тенденции близки. Это приводит к высокой внутригрупповой вариации, которая «маскирует» потенциальные межгрупповые различия при использованном тесте.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь нужно проанализировать другие данные. Представим, что обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Задача — провести оценку результатов A/B-теста на основе:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оценим корректность проведения теста и проанализируем его результаты.

## Цели и задачи исследования


**Цель исследования:**

Оценить, приводит ли внедрение новой версии интерфейса сайта (упрощённый дизайн) к статистически значимому увеличению конверсии зарегистрированных пользователей в покупателей в течение первых семи дней после регистрации, как минимум на 3 процентных пункта по сравнению с текущим интерфейсом.

**Задачи исследования:**

- Проверить корректность проведения A/B-теста: соответствие техническому заданию, равномерность распределения участников по группам, отсутствие пересечений с другими тестами.

- Загрузить и предобработать данные о действиях пользователей, выделить события, относящиеся к участникам теста.

- Ограничить период наблюдения семью днями с момента регистрации для каждого пользователя.

- Рассчитать конверсию в покупку в контрольной (A) и тестовой (B) группах.

- Оценить достаточность объёма выборки для обнаружения ожидаемого эффекта.

- Провести статистическую проверку гипотезы о превосходстве новой версии интерфейса с использованием одностороннего z-теста для разницы пропорций, в том числе с учётом требуемого минимального эффекта в 3 п.п.

- Сформулировать выводы о наличии и практической значимости наблюдаемого эффекта.

## Загрузим данные, оценим их целостность

In [4]:
# Загрузка участников тестов
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
display('Таблица участников тестов:')
display(participants.head())
display(participants.info())

# Загрузка событий
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)
display('Таблица событий:')
display(events.head())
display(events.info())

# Проверка пропусков
display('Пропуски в participants:')
display(participants.isnull().sum())

display('Пропуски в events:')
display(events.isnull().sum())

# Проверка дубликатов
display(f'Дубликатов строк в participants: {participants.duplicated().sum()}')
display(f'Дубликатов строк в events: {events.duplicated().sum()}')

'Таблица участников тестов:'

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


None

'Таблица событий:'

...

user_id    0
group      0
ab_test    0
device     0
dtype: int64

'Пропуски в events:'

user_id            0
event_dt           0
event_name         0
details       538264
dtype: int64

'Дубликатов строк в participants: 0'

'Дубликатов строк в events: 36318'

**Предварительный вывод по этапу загрузки и оценки целостности данных A/B-теста**

- Таблица участников (participants) содержит 14 525 записей без пропусков и дубликатов. Все столбцы (user_id, group, ab_test, device) имеют правильные типы данных.

- Таблица событий (events) содержит 787 286 записей. Обнаружено 36 318 полных дубликатов строк — их необходимо удалить для корректного анализа.

- Столбец details имеет 538 264 пропуска (не заполнен для большинства событий). Поскольку для задачи (расчёт конверсии) детализация не требуется, пропуски в этом столбце не критичны и не повлияют на результаты.

- Типы данных соответствуют ожидаемым: user_id — объект (идентификатор), event_dt — дата/время, event_name — тип события.

Данные пригодны для дальнейшего анализа после удаления дубликатов из таблицы событий и фильтрации по тесту.

## По таблице `ab_test_participants` оценим корректность проведения теста

   Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [5]:
# Выделяем пользователей теста interface_eu_test
test_users = participants[participants['ab_test'] == 'interface_eu_test'].copy()
display(f'Число участников теста interface_eu_test: {test_users.shape[0]}')

# Соответствие требованиям: группы только A и B
actual_groups = set(test_users['group'].unique())
expected_groups = {'A', 'B'}
groups_ok = actual_groups == expected_groups
display(f'Группы в тесте: {actual_groups}')
display(f'Соответствие ТЗ (только A и B): {groups_ok}')

# Равномерность распределения по группам
group_counts = test_users['group'].value_counts()
display('Распределение по группам до удаления пересечений:')
display(group_counts)
chi2, p_chi = chisquare(group_counts)
display(f'Хи-квадрат на равномерность: p-value = {p_chi:.4f}')
if p_chi < 0.05:
    display('Распределение НЕ равномерно (p < 0.05)')
else:
    display('Распределение равномерно (p >= 0.05)')

# Проверка пересечений с другими тестами
user_test_counts = participants.groupby('user_id')['ab_test'].nunique()
multi_test_users = user_test_counts[user_test_counts > 1].index
cross_test_users = test_users[test_users['user_id'].isin(multi_test_users)]
n_cross = cross_test_users['user_id'].nunique()
display(f'Пользователей, участвующих одновременно в нескольких тестах: {n_cross}')
if n_cross > 0:
    display('Обнаружены пересечения – исключаем их из тестовой выборки.')
    # Удаляем пользователей, которые участвуют в нескольких тестах
    test_users = test_users[~test_users['user_id'].isin(multi_test_users)]
    display(f'Число участников после удаления пересечений: {test_users.shape[0]}')
else:
    display('Пересечений с другими тестами нет – условие выполнено.')

'Число участников теста interface_eu_test: 10850'

"Группы в тесте: {'B', 'A'}"

'Соответствие ТЗ (только A и B): True'

'Распределение по группам до удаления пересечений:'

B    5467
A    5383
Name: group, dtype: int64

'Хи-квадрат на равномерность: p-value = 0.4200'

'Распределение равномерно (p >= 0.05)'

'Пользователей, участвующих одновременно в нескольких тестах: 887'

'Обнаружены пересечения – исключаем их из тестовой выборки.'

'Число участников после удаления пересечений: 9963'

Проанализируем данные о пользовательской активности по таблице `ab_test_events`. Оставим только события, связанные с участвующими в изучаемом тесте пользователями.

In [6]:
# Фильтрация событий: оставляем только события участников теста interface_eu_test

test_user_ids = test_users['user_id'].unique()
test_events = events[events['user_id'].isin(test_user_ids)].copy()

display(f'Событий до фильтрации: {events.shape[0]}')
display(f'Событий после фильтрации (только участники теста): {test_events.shape[0]}')

'Событий до фильтрации: 787286'

'Событий после фильтрации (только участники теста): 73815'

Определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации.

In [7]:
# Дата первой регистрации каждого пользователя
first_reg = (
    test_events[test_events['event_name'] == 'registration']
    .groupby('user_id')['event_dt']
    .min()
    .reset_index()
    .rename(columns={'event_dt': 'reg_dt'})
)

# Добавляем дату регистрации к событиям
test_events = test_events.merge(first_reg, on='user_id', how='left')

# Считаем лайфтайм в днях
test_events['lifetime_days'] = (test_events['event_dt'] - test_events['reg_dt']).dt.total_seconds() / (3600 * 24)

# Оставляем только события, произошедшие в первые 7 дней (включая день регистрации)
events_7d = test_events[
    (test_events['lifetime_days'] >= 0) & 
    (test_events['lifetime_days'] <= 7)
].copy()

display(f'Событий после фильтрации по 7 дням: {events_7d.shape[0]}')

'Событий после фильтрации по 7 дням: 63805'

Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [8]:
# Параметры
p1 = 0.30                     # базовая конверсия (контроль)
p2 = 0.33                     # ожидаемая конверсия тестовой группы (p1 + 3 п.п.)
alpha = 0.05                  # уровень значимости (двусторонний)
power = 0.80                  # мощность

# Размер эффекта
effect_size = proportion_effectsize(p2, p1)

# Минимально необходимый размер одной группы для двустороннего теста
n_min = zt_ind_solve_power(
    effect_size=effect_size,
    nobs1=None,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)
display(f'Минимально необходимый размер одной группы (двусторонний тест): {n_min:.0f}')

# Убедимся, что к events_7d присоединена группа
if 'group' not in events_7d.columns:
    events_7d = events_7d.merge(test_users[['user_id', 'group']], on='user_id', how='left')

# Фактические размеры групп
actual_sizes = events_7d.groupby('group')['user_id'].nunique()
display('Фактические размеры групп (уникальные пользователи за 7 дней):')
display(actual_sizes)

# Проверка достаточности
for grp, size in actual_sizes.items():
    if size >= n_min:
        display(f'Группа {grp}: размер достаточен ({size:.0f} >= {n_min:.0f})')
    else:
        display(f'Группа {grp}: размер НЕ достаточен ({size:.0f} < {n_min:.0f})')

'Минимально необходимый размер одной группы (двусторонний тест): 3762'

'Фактические размеры групп (уникальные пользователи за 7 дней):'

group
A    4952
B    5011
Name: user_id, dtype: int64

'Группа A: размер достаточен (4952 >= 3762)'

'Группа B: размер достаточен (5011 >= 3762)'

Рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [9]:
# Общее количество посетителей в каждой группе (уникальные пользователи за 7 дней)
total_visitors = events_7d.groupby('group')['user_id'].nunique()
display('Общее количество посетителей:')
display(total_visitors)

# Количество покупателей (уникальные пользователи с событием purchase)
purchasers = events_7d[events_7d['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()
display('Количество покупателей:')
display(purchasers)

# Конверсия
conversion = purchasers / total_visitors
display('Конверсия в покупку за 7 дней:')
display(conversion)

'Общее количество посетителей:'

group
A    4952
B    5011
Name: user_id, dtype: int64

'Количество покупателей:'

group
A    1377
B    1480
Name: user_id, dtype: int64

'Конверсия в покупку за 7 дней:'

group
A    0.278069
B    0.295350
Name: user_id, dtype: float64

Сделаем предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

После фильтрации событий по семидневному горизонту с момента регистрации и исключения пользователей, одновременно участвовавших в других тестах, получены следующие результаты:

- Общая выборка составила 9963 пользователя, равномерно распределённых по группам (A – 4952, B – 5011).

- Количество пользователей, совершивших покупку в течение первых 7 дней:

  - Контрольная группа A: 1377 покупателей (конверсия 27.8%).

  - Тестовая группа B (новый интерфейс): 1480 покупателей (конверсия 29.5%).

- Наблюдается прирост конверсии в тестовой группе примерно на 1.77 процентных пункта по сравнению с контрольной.

Предварительно можно отметить слабый положительный сдвиг показателя в группе с обновлённым интерфейсом. Окончательный вывод о статистической значимости этого изменения будет сделан после проведения формального z-теста.

## Оценка результатов A/B-тестирования

Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

**Гипотезы A/B-теста**

- **Нулевая гипотеза:** конверсия в покупку в тестовой группе (новый интерфейс, группа B) не превышает конверсию в контрольной группе (старый интерфейс, группа A).

- **Альтернативная гипотеза:** конверсия в тестовой группе (новый интерфейс, группа B) строго больше конверсии в контрольной группе (старый интерфейс, группа A).

Уровень значимости принят равным (alpha = 0.05). Проверка выполняется с помощью одностороннего z-теста для разности пропорций.

In [10]:
# Данные из events_7d
total_visitors = events_7d.groupby('group')['user_id'].nunique()
n_A = total_visitors['A']
n_B = total_visitors['B']

purchases = events_7d[events_7d['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()
k_A = purchases['A']
k_B = purchases['B']

p_A = k_A / n_A
p_B = k_B / n_B

display(f'Группа A: пользователей = {n_A}, покупателей = {k_A}, конверсия = {p_A:.4f}')
display(f'Группа B: пользователей = {n_B}, покупателей = {k_B}, конверсия = {p_B:.4f}')
display(f'Фактическая разница конверсий (B - A): {p_B - p_A:.4f} ({p_B - p_A:.2%})')

# Односторонний z-тест (p_B > p_A)
alpha = 0.05

# Пропорция объединённой выборки
p_pool = (k_A + k_B) / (n_A + n_B)
se_pool = math.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z_stat = (p_B - p_A) / se_pool
# Одностороннее p-value через функцию ошибок
p_val = 0.5 * (1 - math.erf(z_stat / math.sqrt(2)))

display('Результаты одностороннего z-теста:')
display(f'z = {z_stat:.4f}, p-value = {p_val:.4f}')

if p_val < alpha:
    display('Вывод: отвергаем нулевую гипотезу.')
    display('Конверсия в группе с новым интерфейсом (B) статистически значимо выше, чем в контрольной группе (A).')
else:
    display('Вывод: нет оснований отвергнуть нулевую гипотезу.')
    display('Статистически значимого увеличения конверсии в группе B не обнаружено.')

'Группа A: пользователей = 4952, покупателей = 1377, конверсия = 0.2781'

'Группа B: пользователей = 5011, покупателей = 1480, конверсия = 0.2954'

'Фактическая разница конверсий (B - A): 0.0173 (1.73%)'

'Результаты одностороннего z-теста:'

'z = 1.9070, p-value = 0.0283'

'Вывод: отвергаем нулевую гипотезу.'

'Конверсия в группе с новым интерфейсом (B) статистически значимо выше, чем в контрольной группе (A).'

Опишем выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

**Выводы по результатам A/B-тестирования**

1. **Статистическая значимость**  
   Односторонний z-тест показал, что конверсия в группе B (новый интерфейс) статистически значимо выше, чем в группе A (p-value = 0.0283). Таким образом, нулевая гипотеза \(H_0: p_B \le p_A\) отвергается на уровне значимости 5%. Обновлённый дизайн действительно повышает вероятность совершения покупки в течение первых семи дней после регистрации.

2. **Величина эффекта и ожидаемый прирост**  
   Фактический прирост конверсии составил **1.73 процентных пункта** (с 27.81% до 29.54%). Это ниже запланированного в техническом задании минимального эффекта в **3 п.п.** Следовательно, **ожидаемый бизнес‑эффект не был достигнут**, несмотря на наличие статистически значимых различий.

3. **Корректность теста и ограничения**  
   Исходно в тесте присутствовало 887 пользователей, одновременно участвовавших в других экспериментах, что нарушало чистоту сравнения. Эти участники были исключены из финальной выборки. После фильтрации группы оказались сбалансированы по размеру (A: 4952, B: 5011) и соответствовали требованиям достаточности выборки (минимально необходимое количество — 3762 в каждой). Таким образом, итоговый анализ проведён на корректных данных, однако факт первоначального пересечения всё же требует осторожности в интерпретации и рекомендации повторить тест в изолированной среде.

4. **Общий итог**  
   Новый интерфейс статистически значимо увеличивает конверсию, но прирост не достигает установленного порога 3 п.п. Результат может считаться умеренно положительным, но с точки зрения поставленной бизнес‑цели тест не подтвердил достаточную эффективность изменений. Рекомендуется либо доработать дизайн, чтобы усилить эффект, либо пересмотреть целевой показатель, если фактический прирост признаётся экономически оправданным.